# Overview
Author – Thomas Hervey 

`geopandas_tutorial.ipynb` is the main file that I created.

This is a project to learn `geopandas` as instructed by Matt Forester (video: https://www.youtube.com/watch?v=0mWgVVH_dos). The tutorial's data and sample notebooks are in the folders. GeoPandas organizes pandas by adding `index` and `geometry` columns and allows spatial analysis operations on vector data.

## 0. Setup

I am using `git-flow` for version control, `pyenv` for managing Python versions (3.10 for this project) and `poetry` for managing dependencies. I am running this code as a notebook in an IDE such as VS Code or Windsurf.

- First, initialize `git-flow` with 
    ```bash
    git flow init
    ```

- Second, let's create a new poetry environment:
    ```bash
    poetry init --name geopandas-course --description "GeoPandas course materials" --author "Thomas" --python "^3.9" --dependency geopandas --dependency jupyter --dependency notebook --no-interaction
```

GeoPandas organizes pandas by adding `index` and `geometry` columns. 
    ```

- Third, install dependencies with:
    ```bash
    poetry install
    ```

- Fourth, let's install the IPython kernel for the Poetry environment:
    ```bash
    poetry run python -m ipykernel install --user --name geopandas-course --display-name "Python (GeoPandas Course)"
    ```

- Last, open the Kernel option in the IDE and select the Python kernel named `geopandas-course-...`

# 1. Introduction to GeoDataFrames

## Creating a Simple Data Structure

In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
# Data for five large global cities (one from each continent)
data = {
    "Name": ["New York City", "São Paulo", "Tokyo", "Lagos", "Sydney"],
    "Population": [8419600, 12325232, 13929286, 15000000, 5312163],  # Approximate populations
    "Latitude": [40.7128, -23.5505, 35.6895, 6.5244, -33.8688],
    "Longitude": [-74.0060, -46.6333, 139.6917, 3.3792, 151.2093]
}
print(type(data))
print(data["Name"])

In [ ]:
cities_df = pd.DataFrame(data)
print(cities_df[['Name', 'Population']])

## GeoPandas: GeoDataFrames & GeoSeries

In [ ]:
# Create a geopandas GeoDataFrame
gdf = gpd.GeoDataFrame(
    cities_df,
    # Create the geometry column using x,y columns
    geometry=gpd.points_from_xy(
        x=cities_df['Longitude'],
        y=cities_df['Latitude'],
        z=None,
        crs="EPSG:4326"  # NOTE: you can also use a WKT text
        )
    )

print(gdf.info())
print(gdf["geometry"])

In [ ]:
# Render the geometry of the first row (uses the `shapely` library for rendering geometries)
gdf['geometry'][0]

### Polygons

In [ ]:
# Create a WKT text for a polygon of Wisconsin

wisconsin = 'POLYGON((-90.6303 42.5116,-87.0241 42.4954,-87.1477 43.3721,-87.1230 43.6838,-86.9911 44.1694,-86.8579 44.4828,-86.6876 44.8792,-86.2523 45.2343,-86.7535 45.4418,-87.1024 45.4438,-87.1724 45.3531,-87.3193 45.2401,-87.4100 45.1995,-87.4443 45.0784,-87.6379 45.0987,-87.6956 45.2217,-87.6462 45.3367,-87.6956 45.3869,-87.8549 45.3425,-87.7945 45.4813,-87.7945 45.5679,-87.7876 45.6764,-87.9936 45.7934,-88.1145 45.8077,-88.0733 45.8728,-88.1982 45.9521,-88.5498 46.0122,-88.7036 45.9970,-88.8451 46.0380,-89.0909 46.1389,-90.1195 46.3412,-90.4147 46.5664,-89.9615 47.2876,-90.6578 47.3100,-92.0352 46.7079,-92.2948 46.5928,-92.2975 46.0761,-92.3579 46.0132,-92.5227 45.9769,-92.7466 45.8135,-92.8564 45.6140,-92.7754 45.5477,-92.6587 45.4293,-92.7136 45.3029,-92.7466 44.9648,-92.7686 44.7877,-92.6422 44.6530,-92.3552 44.5504,-92.2632 44.4652,-92.0654 44.4024,-91.8745 44.2580,-91.6383 44.0600,-91.4941 43.9612,-91.3184 43.8296,-91.2524 43.6440,-91.2305 43.5167,-91.2085 43.3342,-91.0547 43.2932,-91.1865 43.1651,-91.1536 43.0257,-91.1426 42.9082,-91.0547 42.7692,-90.7910 42.6915,-90.7031 42.6340,-90.6303 42.5096,-90.6303 42.5116))'

data = {
    "Name": ["Wisconsin"],
    "Population": [5911000],  # Approximate populations
    "geometry": [wisconsin],
}

gdf2 = gpd.GeoDataFrame(data)
gdf2.head()

In [ ]:
# This is just a string (it won't render), not an actual geometry because we didn't define what the geometry type is
gdf2['geometry'][0]

In [ ]:
# So, create a NEW dataframe but using WKT as the geometry type

gdf3 = gpd.GeoDataFrame(
    data,
    geometry=gpd.GeoSeries.from_wkt(data['geometry'])
)

gdf3['geometry'][0]

In [ ]:
type(gdf3['geometry'][0])

# 2. GDF Visualizations

### Basic plotting using common plotting libraries

In [ ]:
# Plot the geometry
gdf.plot()

In [ ]:
# Create a nicer population plot using Matplotlib

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(ax=ax, color='red', markersize=50)
for x, y, label in zip(gdf.geometry.x, gdf.geometry.y, gdf["Population"]):
    ax.text(x, y, label, fontsize=8, color='black', ha='right')
plt.title("Point Dataset with Labels")
plt.show()

In [ ]:
# Set the Dataframe CRS
gdf = gdf.set_crs(epsg=4326) # NOTE: There are many other styles e.g., 'EPSG:4326'
gdf.crs

In [ ]:
# Quickly explore the data on a map. This requires `folium` and `mapclassify` deps.
gdf.explore()

In [ ]:
# Alterantively, use Leafmap (not leaflet directly) to visualize
import leafmap

m = leafmap.Map(center=(0, 0), zoom=2)
m

In [ ]:
# Add a DataFrame as a layer to the leafmap map
m.add_gdf(gdf, layer_name="Points")

In [ ]:
# Add a basemap
m.add_basemap("Esri.WorldImagery")  # Satellite imagery

In [ ]:
# Add a pop-up
m = leafmap.Map(center=[0, 0], zoom=2)
m.add_gdf(gdf, layer_name="Points", tooltips=['Name', 'Population'])
m

In [ ]:
# Alternatively, use Lonboard for visualization

from pathlib import Path

import geopandas as gpd
import pandas as pd
import shapely
from palettable.colorbrewer.diverging import BrBG_10 # Import a color bar pallet
from sidecar import Sidecar # An output widget

# lonboard serves fast vector viz in Jupyter (https://github.com/developmentseed/lonboard)
from lonboard import Map, ScatterplotLayer
from lonboard.colormap import apply_continuous_cmap

### Using `lonboard` to visualize a large GeoDataFrame as a layer

In [ ]:
# An example data file of web speed test performance 

url = "https://ookla-open-data.s3.us-west-2.amazonaws.com/parquet/performance/type=mobile/year=2019/quarter=1/2019-01-01_performance_mobile_tiles.parquet"

In [ ]:
local_path = Path("tutorial_data/internet-speeds.parquet")

# Load the existing parquet file
if local_path.exists():
    gdf = gpd.read_parquet(local_path)

# Or create a new parquet file
else:
    columns = ["avg_d_kbps", "tile"] # Name the columns
    df = pd.read_parquet(url, columns=columns) # Create a pd df from parquet file

    tile_geometries = shapely.from_wkt(df["tile"]) # Set up a tile geometry
    tile_centroids = shapely.centroid(tile_geometries) # Set the centroids from the geom
    
    # Create a gdf using the important data, tile_centroids as geometry
    gdf = gpd.GeoDataFrame(df[["avg_d_kbps"]], geometry=tile_centroids, crs="EPSG:4326")
    gdf.to_parquet(local_path) # Save gdf to parquet

In [ ]:
gdf.head()

In [ ]:
sidecar = Sidecar()

In [ ]:
# Create a lonboard layer (ScatterplotLayer) by loading a GeoPandas gdf
layer = ScatterplotLayer.from_geopandas(gdf)

# Create a lonboard map using the layer
m = Map(layer, _height=800)
# with sidecar:
#     display(m)
m

### Expand our visualization by showing average speed compared to surroundings

In [ ]:
layer.get_fill_color = [0, 0, 200, 200]

min_bound = 5000
max_bound = 50000
download_speed = gdf["avg_d_kbps"]
# Calculate a noralized download speed
normalized_download_speed = (download_speed - min_bound) / (max_bound - min_bound)

In [ ]:
normalized_download_speed

In [ ]:
# Change the fill color to use the normalized speed
layer.get_fill_color = apply_continuous_cmap(
    normalized_download_speed, BrBG_10, alpha=0.7
)

In [ ]:
# Change the point size based on normalized speed
layer.get_radius = normalized_download_speed * 200
layer.radius_units = "meters"
layer.radius_min_pixels = 0.5

In [ ]:
m

# 3. Loading Geospatial Data with GeoPandas

### Loading Shp, GeoJSON, and FGB (flat geobuff) from files & remote sources

In [ ]:
# Import necessary libraries
import geopandas as gpd
import pandas as pd

In [ ]:
# Load in a shapefile (pedestrian zones), and create a gdf
shapefile_path = "tutorial_data/Pedestrian Zone Shapefile (Tabular)_20241220/geo_export_a7bb075a-41dc-445f-8244-8430e90a8dde.shp"
shapefile_gdf = gpd.read_file(shapefile_path) # Read the file directly
shapefile_gdf.explore() # Explore it on a map

In [ ]:
# Load in a geojson file (NYC neighborhoods), and create a gdf, and explore it
geojson_fp_path = "https://raw.githubusercontent.com/HodgesWardElliott/custom-nyc-neighborhoods/refs/heads/master/custom-pedia-cities-nyc-Mar2018.geojson"
geojson_gdf = gpd.read_file(geojson_fp_path)

geojson_gdf.explore()

In [ ]:
# Load in a geopackage (major USA rivers), and create a gdf
gpkg_fp_path = "https://ngageoint.github.io/GeoPackage/examples/rivers.gpkg"
gpkg_gdf = gpd.read_file(gpkg_fp_path)
gpkg_gdf.explore()

In [ ]:
# Load in a Flat Geobuff (USA counties), and create a gdf
fgb_fp = "https://github.com/flatgeobuf/flatgeobuf/raw/refs/heads/master/test/data/UScounties.fgb"
fgb_gdf = gpd.read_file(fgb_fp)
fgb_gdf.explore()

### Connect to PostGIS data

In [ ]:
import geopandas as gpd
import pandas as pd
from sqlalchemy import create_engine

# Load the password from a .env file
from dotenv import load_dotenv
import os
load_dotenv()


# Define PostGIS connection parameters
host = "localhost"           # e.g., "localhost" or your database IP
database = "YOUR_DB_NAME"  # update to your local DB name   # Database name
user = "YOUR_USERNAME"  # update to your DB username           # Username
password = os.getenv("PG_PASSWORD")
port = "5432"                # Default PostgreSQL port

# SQLAlchemy connection string for PostGIS
postgis_connection = f"postgresql://{user}:{password}@{host}:{port}/{database}"

# Create SQLAlchemy engine
engine = create_engine(postgis_connection)

# Define SQL query to read spatial data
query = "SELECT * FROM subway_entrances"

# Load the PostGIS table (subway_entrances table that I had previously added to postgis) into a GeoDataFrame
postgis_gdf = gpd.read_postgis(query, con=engine, geom_col="geom", crs="EPSG:4326")

postgis_gdf.head()

### Connect to DuckDB in-memory instance

In [ ]:
import duckdb

# Initialize a DuckDB (an in-memory analytical database) connection
con = duckdb.connect()

In [ ]:
# Install and load the DuckDB spatial library
con.query('INSTALL spatial')
con.query('LOAD spatial')

In [ ]:
# Define a DuckDB SQL query for data from a .zip (citibike trips)
duckdb_query = """
SELECT *, ST_AsText(ST_Point(column09, column08)) as geometry
FROM read_csv('https://s3.amazonaws.com/tripdata/202406-citibike-tripdata.zip', ignore_errors=true)
"""

In [ ]:
# Execute DuckDB query and convert to a Pandas DataFrame
duckdb_df = con.query(duckdb_query).to_df()

# The geom column lay look a bit odd, but we can transform it into a gdf.
duckdb_df.head(5)

In [ ]:
# Create a gdf from the DuckDB pd df
duckdb_gdf = gpd.GeoDataFrame(
    duckdb_df,
    geometry=gpd.GeoSeries.from_wkt(duckdb_df["geometry"]),
    crs="EPSG:4326"
)

duckdb_gdf.info()

In [ ]:
# Map 100 records
duckdb_gdf.head(100).explore()

In [ ]:
# Alternatively, define a DuckDB SQL read 10 rows from a .geojson (NYC neighborhoods)
nbh_query = """
SELECT *
FROM ST_Read('# NOTE: update the path below to your local file location
# data/Projects/web_courses/2025_03_geopandas/tutorial_data/custom_nyc_neighborhoods.geojson')
limit 10
"""

In [ ]:
# Execute DuckDB query and convert to a Pandas DataFrame
duckdb_df = con.query(nbh_query).to_df()
duckdb_df.head(5)

In [ ]:
# Again, alternatively, define a DuckDB SQL query that:
# - Counts the number of 'trips'
# - Return the neighborhood name and geometry
# - Run an intersection between the neighborhood geometry and starting geometry of bike trip
# - Select for a specific date
# - Group the data

nbh_query = """
SELECT 
    count(b.column00) as count,
    n.neighborhood,
    ST_AsText(n.geom) as geom
FROM 
    ST_Read('# NOTE: update the path below to your local file location
# data/Projects/web_courses/2025_03_geopandas/tutorial_data/custom_pedia_cities_nyc.geojson') n
JOIN 
    read_csv('https://s3.amazonaws.com/tripdata/202406-citibike-tripdata.zip', ignore_errors=true) b
ON 
    ST_Intersects(n.geom, ST_Point(column09, column08))
WHERE 
    CAST(column02 AS DATE) = DATE '2024-06-15'
GROUP BY 
    n.neighborhood, n.geom
"""

In [ ]:
# Execute the spatial join query, and convert the results to a pd df
nbh_df = con.query(nbh_query).to_df()

In [ ]:
# Create a gdf from the df using the 'geom' column
join_gdf = gpd.GeoDataFrame(
    nbh_df,
    geometry=gpd.GeoSeries.from_wkt(nbh_df["geom"]),
    crs="EPSG:4326"
)
join_gdf.head()

In [ ]:
join_gdf.explore()

### Load (geo)Parquet

In [ ]:
# Load a geo parquet file
gdf = gpd.read_parquet('data/es_cn.parquet')
gdf.info()

In [ ]:
gdf.head(100).explore()

# 4. Spatial Functions in GeoPandas

In [ ]:
import geopandas as gpd

In [ ]:
# Load data
schools = gpd.read_file("tutorial_data/nyc/SchoolPoints_APS_2024_08_28 (1)/SchoolPoints_APS_2024_08_28.shp")

subways = gpd.read_file("tutorial_data/nyc/nyc_subway_entrances/nyc_subway_entrances.shp")

bike_paths = gpd.read_file("tutorial_data/nyc/New York City Bike Routes_20241223.geojson")

neighborhoods = gpd.read_file("https://raw.githubusercontent.com/HodgesWardElliott/custom-nyc-neighborhoods/refs/heads/master/custom-pedia-cities-nyc-Mar2018.geojson")

parks = gpd.read_file("tutorial_data/nyc/Parks Properties_20241223.geojson")

In [ ]:
schools.explore()

In [ ]:
subways.explore()

In [ ]:
bike_paths.explore()

In [ ]:
parks.explore()

In [ ]:
# Set the CRS for all the layers
schools = schools.to_crs("EPSG:3857")
subways = subways.to_crs("EPSG:3857")
bike_paths = bike_paths.to_crs("EPSG:3857")
neighborhoods = neighborhoods.to_crs("EPSG:3857")
parks = parks.to_crs("EPSG:3857")

In [ ]:
import pandas as pd
pd.options.display.float_format = '{:.4f}'.format # Change decimal place

### Calculate certain geometries from Geoseries

In [ ]:
# 1. GeoSeries.area – Calculate the area of neighborhoods

neighborhoods["area"] = neighborhoods.geometry.area # Calculate the area, save as a new column
neighborhoods[['neighborhood', 'area']]

In [ ]:
neighborhoods.info(verbose=True)

In [ ]:
# 2. GeoSeries.boundary – Map and print the bounding boxes of parks

parks["boundary"] = parks.geometry.boundary # Calculate the boundary, and save as a new column
parks_mapped = parks['boundary'].to_crs('EPSG:4326') # Creating a new df for boundary so that we can map it (we have to map in 4326)
parks_mapped.explore()

In [ ]:
# Print the bounding box of each record
parks_mapped.geometry.bounds

In [ ]:
# Print the bounding box of the entire dataframe
parks_mapped.geometry.total_bounds

In [ ]:
# 3. Geoseries length – Print the lengths of bike paths

# Calculate the length, save as a new field
bike_paths["length"] = bike_paths.geometry.length
# Sort the bike path segments by length
bike_paths[['segmentid', 'length']].sort_values(by=['length'], ascending=False)

In [ ]:
# Reproject the bike paths to get values in feet (not meters)

# Create a df of the bike paths in a New York CRS
paths_2263 = bike_paths.to_crs('EPSG:2263')

# Calculate the length, save as a new field
paths_2263["length"] = paths_2263.geometry.length
# Sort the bike path segments by length
paths_2263[['segmentid', 'length']].sort_values(by=['length'], ascending=False)

In [ ]:
# 5. Map a filtered record

# Map the longest bike path (filtering by a specific segment ID)
bike_paths[
    bike_paths['segmentid'].astype(str) == '299972.0'
    ].to_crs(
        'EPSG:4326'
        ).explore()

In [ ]:
# 6. Calculate the shortest distance from schools to the nearest subway entrance

# Save a new column
schools["nearest_subway_distance"] = schools.geometry.apply(
    # Function that, given a school gets the min distance between subways (entrances) and the school
    lambda school: subways.distance(school).min()
)

# Sort schools by their nearest subway (entrances) distances
schools[['Name', 'nearest_subway_distance']].sort_values(by=['nearest_subway_distance'], ascending=True)

In [ ]:
# 7. Create centroids from neighborhood geometry

neighborhoods.geometry.centroid.explore()

In [ ]:
# 8. Map a 50 meter buffer around 100 bike paths

bike_paths.head(100).geometry.buffer(50).explore()

In [ ]:
# 9. Calculate intersections between bike paths and parks

parks["intersects_bike_path"] = parks.geometry.apply(
    # Function that finds any bike paths that intersect a given park
    lambda park: bike_paths.geometry.intersects(park).any()
)

# Then, map the parks where there is a bike path
parks[parks['intersects_bike_path'] == True].explore()

In [ ]:
# 10. Envelope, and Minimum-bounding circle of parks

parks.geometry.envelope.explore()

parks.geometry.minimum_bounding_circle().explore()

In [ ]:
# 11. Convex Hull of neighborhoods

neighborhoods.geometry.convex_hull.explore()

In [ ]:
# 12. Simplify the neighborhood geometry

neighborhoods.geometry.simplify(tolerance=0.001).explore()

In [ ]:
# 13. Calculate and map the parks that overlay neighborhoods

parks_neighborhoods_overlay = neighborhoods.overlay(parks, how="intersection")

parks_neighborhoods_overlay.explore()

### Lower-level geometric functions

In [ ]:
# 1. Translate points by 10000 ft

subways["translated"] = subways.geometry.translate(xoff=10000, yoff=10000)
subways.translated.explore()

In [ ]:
# 2. Check if geometry is valid

parks.geometry.is_valid

In [ ]:
# 3. Get the exterior of neighborhood polygons

neighborhoods.geometry.exterior.explore()

In [ ]:
# 4. Print the geometry types of bike paths records

bike_paths.geometry.geom_type

In [ ]:
# 5. Calculate new x, y columns
schools["x"] = schools.geometry.to_crs(epsg=4326).x
schools["y"] = schools.geometry.to_crs(epsg=4326).y
schools.info()

In [ ]:
# 6. Calculate the neighborhoods that contain parks

# Create a new column
neighborhoods["contains_parks"] = neighborhoods.geometry.apply(
    # Calculate which parks contain any neighborhood given a neighborhood
    lambda neighborhood: parks.geometry.apply(
        # Calculate which neighborhoods contains a given park
        lambda park: neighborhood.contains(park)
    ).any()
)

# Map the neighborhoods that do not contain parks (NOTE: this is containment, so some are entirely parks)
neighborhoods[neighborhoods['contains_parks'] == False].explore()

In [ ]:
# 7. Calculate bike paths that cross over into a park (starts outside a park, and crosses the boundary into a park)

bike_paths["crosses_parks"] = bike_paths.geometry.apply(
    lambda path: path.crosses(parks.geometry).any()
)
bike_paths[bike_paths['crosses_parks'] == True].explore()

In [ ]:
# 8. Find all parks that are disjointed from all neighborhoods

parks["disjoint_neighborhoods"] = parks.geometry.apply(
    lambda park: park.disjoint(neighborhoods.geometry).all()
)
parks[parks['disjoint_neighborhoods'] == True].explore()

In [ ]:
# 9. How many schools are within 500 meters of a subway entrance?

# Calculate any schools within 500 meters of all subways
def schoolsWithin500(school):
    return school.distance(subways.geometry).any() <= 500

# Create a new column using the schools gemoetry
schools["within_500m_subway"] = schools.geometry.apply(
    schoolsWithin500
)

# unionize all rows
subway_union = subways.geometry.union_all()

# Get just the schools that are within 500 meters
schools["within_500m_subway"] = schools.geometry.distance(subway_union) <= 500

# Map the subset of results
schools[schools["within_500m_subway"] == True].explore()

In [ ]:
# 10. Which parks overlap a neighborhood?

parks["overlaps_neighborhoods"] = parks.geometry.apply(
    lambda park: park.overlaps(neighborhoods.geometry).any()
)

parks[parks["overlaps_neighborhoods"] == True].explore()

In [ ]:
# 11. Semetrical difference, parks that are clipped out of a neighborhood

parks['geometry'].symmetric_difference(neighborhoods['geometry']).explore()

# 5. Project 1 – Spatial Joins

In [ ]:
import geopandas as gpd

In [ ]:
# Load data

schools = gpd.read_file("data/nyc/SchoolPoints_APS_2024_08_28 (1)/SchoolPoints_APS_2024_08_28.shp")
subways = gpd.read_file("data/nyc/nyc_subway_entrances/nyc_subway_entrances.shp")
bike_paths = gpd.read_file("data/nyc/New York City Bike Routes_20241223.geojson")
neighborhoods = gpd.read_file("https://raw.githubusercontent.com/HodgesWardElliott/custom-nyc-neighborhoods/refs/heads/master/custom-pedia-cities-nyc-Mar2018.geojson")
parks = gpd.read_file("data/nyc/Parks Properties_20241223.geojson")

In [ ]:
# Set CRS for data

schools = schools.to_crs("EPSG:3857")
subways = subways.to_crs("EPSG:3857")
bike_paths = bike_paths.to_crs("EPSG:3857")
neighborhoods = neighborhoods.to_crs("EPSG:3857")
parks = parks.to_crs("EPSG:3857")

In [ ]:
# 1. Set a spatial index on our data

sindex = neighborhoods.sindex

In [ ]:
# 2. Find schools completely within neighborhoods using an 'inner within' join

schools_in_neighborhoods = gpd.sjoin(schools, neighborhoods, how="inner", predicate="within")

In [ ]:
schools_in_neighborhoods.head()

In [ ]:
# 3. Aggregate the results: count schools per neighborhood and include geometry

# A. Create a new data frame by grouping neighborhood, aggregating school count, and resetting the index
schools_count_by_neighborhood = schools_in_neighborhoods\
    .groupby('neighborhood')\
    .agg(school_count=('index_right', 'count'))\
    .reset_index()

# B. Merge with the neighborhoods dataframe to preserve neighborhood geometry, and fill in n/a values
schools_count_by_neighborhood = neighborhoods\
    .merge(schools_count_by_neighborhood, on='neighborhood', how='left')\
    .fillna({'school_count': 0})

# C. Convert to a GeoDataFrame
schools_count_by_neighborhood = gpd.GeoDataFrame(schools_count_by_neighborhood, geometry='geometry', crs=neighborhoods.crs)

schools_count_by_neighborhood

# 6. Final Project

Creating an access index by seeing how accessible different neighborhoods are

In [ ]:
import geopandas as gpd
import h3pandas
from shapely.geometry import Point, Polygon
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pandas as pd

In [ ]:
# Load data

schools = gpd.read_file("tutorial_data/nyc/SchoolPoints_APS_2024_08_28 (1)/SchoolPoints_APS_2024_08_28.shp")
subways = gpd.read_file("tutorial_data/nyc/nyc_subway_entrances/nyc_subway_entrances.shp")
bike_paths = gpd.read_file("tutorial_data/nyc/New York City Bike Routes_20241223.geojson")
neighborhoods = gpd.read_file("https://raw.githubusercontent.com/HodgesWardElliott/custom-nyc-neighborhoods/refs/heads/master/custom-pedia-cities-nyc-Mar2018.geojson")
parks = gpd.read_file("tutorial_data/nyc/Parks Properties_20241223.geojson")

In [ ]:
# Set data CRS

schools = schools.to_crs("EPSG:3857")
subways = subways.to_crs("EPSG:3857")
bike_paths = bike_paths.to_crs("EPSG:3857")
neighborhoods = neighborhoods.to_crs("EPSG:3857")
parks = parks.to_crs("EPSG:3857")

## Anayze Neighborhoods

Understand the access to different features within each neighborhood (ex. schools, length of bike paths, number of subways, area of parks)

In [ ]:
# Count features intersecting the neighborhood boundary
def count_intersect_with_neighborhood(neighborhood_geometry):
    num_schools = schools[schools.geometry.intersects(neighborhood_geometry)].shape[0]
    
    num_subways = subways[subways.geometry.intersects(neighborhood_geometry)].shape[0]
    
    bike_path_length = bike_paths[bike_paths.geometry.intersects(neighborhood_geometry)].length.sum()
    
    park_area = parks[parks.geometry.intersects(neighborhood_geometry)].area.sum()

    return num_schools, num_subways, bike_path_length, park_area

In [ ]:
# 1. Apply analysis counting intersections to each neighborhood

# Create new columns coming back from the function that accepts the neighborhood geometry
neighborhoods[['num_schools', 'num_subways', 'bike_path_length', 'park_area']] = neighborhoods.geometry.apply(
    lambda geom: pd.Series(count_intersect_with_neighborhood(geom))
)

In [ ]:
neighborhoods.head(3)

In [ ]:
# 2. Normalize results (put values on 0 to 1 scale)

scaler = MinMaxScaler()
columns_to_normalize = ['num_schools', 'num_subways', 'bike_path_length', 'park_area']

# Transform using fit_transform
neighborhoods[columns_to_normalize] = scaler.fit_transform(neighborhoods[columns_to_normalize])

neighborhoods.head(3)

In [ ]:
# 3. Aggregate results using touching neighborhoods (an additional normalization step to normalize to neighborhood area)

# Calculate the neighborhoods that touch tha aggregate
def aggregate_touching_neighborhoods(neighborhood_index):

    # Get the geometry of the incoming neighborhood ID
    current_geometry = neighborhoods.loc[neighborhood_index, 'geometry']

    # Get the index of the neighborhoods that touch our incoming neighborhood
    touching_indices = neighborhoods[neighborhoods.geometry.touches(current_geometry)].index

    # If there is 1+ touching neighborhood
    if not touching_indices.empty:
        # Get the average of the normalized columns of those touching
        neighbor_values = neighborhoods.loc[touching_indices, columns_to_normalize].mean()
    else:
        # Otherwise just get the incoming neighborhood's columns
        neighbor_values = neighborhoods.loc[neighborhood_index, columns_to_normalize]

    return neighbor_values


# Apply aggregation to each neighborhood
neighborhoods[columns_to_normalize] = neighborhoods.index.to_series().apply(
    lambda idx: aggregate_touching_neighborhoods(idx)
)

# Final normalization (0 to 1 scale)
neighborhoods[columns_to_normalize] = scaler.fit_transform(neighborhoods[columns_to_normalize])

neighborhoods.head(3)


In [ ]:
# 4. Create a final single index score

neighborhoods['index_score'] = neighborhoods['num_schools'] + neighborhoods['num_subways'] + neighborhoods['bike_path_length'] + neighborhoods['park_area'] 

In [ ]:
import leafmap

m = leafmap.Map()
m.add_data(
    neighborhoods, column="index_score", scheme="Quantiles", cmap="Blues", legend_title="Index"
)
m

In [ ]:
# Save the results
neighborhoods.to_file("tutorial_data/neighborhood_access_index.geojson", driver="GeoJSON")

## Redo this with H3 Grid cells

Create a 9 resolution grid of h3 cells for our neighborhoods

In [ ]:
# 1. Create h3 grid

neighborhoods = neighborhoods.to_crs('EPSG:4326')

resolution = 9  # Adjust resolution as needed

# Create a gdf using h3 grid
gdf_h3 = neighborhoods.h3.polyfill(resolution) 

gdf_h3.head()

In [ ]:
# 2. Explode the resolutions into their own rows

# Create the gfd, but explode the rows for each h3 resolution level
gdf_h3 = neighborhoods.h3.polyfill(resolution, explode=True)
gdf_h3.head()

In [ ]:
# 3. Use grid for index, then convert those into boundaries

# Finding any rows that are not null, and set the index to that h3 value
new_gdf_h3 = gdf_h3[gdf_h3['h3_polyfill'].isnull() == False].set_index('h3_polyfill')
new_gdf_h3.index.name = None

# Convert the h3 index values into boundaries
new_gdf_h3 = new_gdf_h3.h3.h3_to_geo_boundary()

new_gdf_h3.explore()

### Crete an H3 Project

In [ ]:
# Set to the correct CRS

gdf_h3_proj = gdf_h3.to_crs('EPSG:3857')

In [ ]:
# Analysis for each hex cell
def analyze_access(hex_geometry):
    # Buffer hex geometry
    buffer_1600m = hex_geometry.buffer(1600)
    buffer_800m = hex_geometry.buffer(800)

    # Count features within buffers
    num_schools = schools[schools.geometry.intersects(buffer_1600m)].shape[0]
    num_subways = subways[subways.geometry.intersects(buffer_1600m)].shape[0]
    bike_path_length = bike_paths[bike_paths.geometry.intersects(buffer_1600m)].length.sum()
    park_area = parks[parks.geometry.intersects(buffer_800m)].area.sum()

    return num_schools, num_subways, bike_path_length, park_area

In [ ]:
# For each h3 cell, calculate the buffered intersections of each accessible amenity
gdf_h3_proj[['num_schools', 'num_subways', 'bike_path_length', 'park_area']] = gdf_h3_proj.geometry.apply(
    lambda hex_geom: pd.Series(analyze_access(hex_geom))
)

gdf_h3_proj.head()

In [ ]:
# Create a column that is the h3 index.
gdf_h3_proj['h3_index'] = gdf_h3_proj.index

In [ ]:
gdf_h3_proj['h3_index']

In [ ]:
gdf_h3_proj['h3_index']

In [ ]:
gdf_h3_proj['h3_index']

### Run normalization

In [ ]:
import h3

# Normalize results
scaler = MinMaxScaler()
normalized_columns = ['num_schools', 'num_subways', 'bike_path_length', 'park_area']
gdf_h3_proj[normalized_columns] = scaler.fit_transform(gdf_h3_proj[normalized_columns])

In [ ]:
print("H3 index type:", type(gdf_h3_proj['h3_index'].iloc[0]))
print("\nFirst few H3 indexes:")
print(gdf_h3_proj['h3_index'].head())

In [ ]:
from h3.api.basic_int import grid_disk

def aggregate_neighbors(h3_index):
    # Assuming h3_index is already an integer
    neighbors = grid_disk(h3_index, 2)  # 2-ring disk
    neighbor_values = gdf_h3_proj[gdf_h3_proj['h3_index'].isin(neighbors)][normalized_columns].mean()
    return neighbor_values


gdf_h3_proj[normalized_columns] = gdf_h3_proj['h3_index'].apply(
    lambda h3_index: aggregate_neighbors(h3_index)
)

In [ ]:
# Final normalized analysis
gdf_h3_proj[normalized_columns] = scaler.fit_transform(gdf_h3_proj[normalized_columns])

gdf_h3_proj.head(3)

### Calculate the final index score

In [ ]:
# Calculate the index score
gdf_h3_proj['index_score'] = gdf_h3_proj['num_schools'] + gdf_h3_proj['num_subways'] + gdf_h3_proj['bike_path_length'] + gdf_h3_proj['park_area'] 

In [ ]:
# Change the projection so we can map it
gdf_h3_map = gdf_h3_proj.to_crs('EPSG:4326')

In [ ]:
import leafmap

m = leafmap.Map()
m.add_data(
    gdf_h3_map, column="index_score", scheme="Quantiles", cmap="Blues", legend_title="Index"
)
m